In [ ]:
# Simple API call
from anthropic import Anthropic
from anthropic.types import TextBlock
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(usecwd=True, raise_error_if_not_found=True)  # fail loudly if missing
load_dotenv(env_path, override=True)  # notebook only: your edited .env wins over stale values

client = Anthropic()  # reads ANTHROPIC_API_KEY
model = "claude-haiku-4-5"

message = client.messages.create(
    model=model,
    max_tokens=500,
    messages=[
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ],
)

text_reponse = "".join(block.text for block in message.content if isinstance(block, TextBlock))
print(text_reponse)
print(f"tokens: in={message.usage.input_tokens} out={message.usage.output_tokens}")
print(f"Stop reason: {message.stop_reason}")


In [ ]:
# Multi-Turn conversations
from anthropic import Anthropic
from anthropic.types import TextBlock
from dotenv import find_dotenv, load_dotenv

env_path = find_dotenv(usecwd=True, raise_error_if_not_found=True)
print(f".env file found: {env_path}")

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"


def add_user_message(messages: list[dict[str, str]], text: str) -> None:
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages: list[dict[str, str]], text: str) -> None:
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages: list[dict[str, str]], system: str | None = None, **more_params) -> TextBlock:
    # Temprature, top_p and top_k are removed from Python SDK since v1.0
    params = {
        "model": model,
        "max_tokens": 500,
        "messages": messages,
    }
    if system:
        params["system"] = system

    params.update(**more_params)

    message = client.messages.create(**params)
    print(f"\n    tokens: in={message.usage.input_tokens} out={message.usage.output_tokens}")
    print(f"    Stop reason: {message.stop_reason}")
    for block in message.content:
        if isinstance(block, TextBlock):
            return block
    raise RuntimeError("No text was found in message object.")


system = """
You are a patient quant computer system engineer tutor.
Do not directly answer a student's questions.
Guide them to solution step by step.
"""

# Start with an empty message list
messages: list[dict[str, str]]= []

# Add the initial user question
add_user_message(messages, "Define quantum computing in one sentence")

# Get Claude's response -- Without system prompt
answer = chat(messages)
print(f"=====Answer WITHOUT System Prompt: {answer.text}")

# Add Claude's response to the conversation history
add_assistant_message(messages, answer.text)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context -- With system prompt
final_answer = chat(messages, system=system)
print(f"=====Final Answer WITH System Prompt: {final_answer.text}")


In [ ]:
# Test Claude API Stream (client receives chuncks of text instead of waiting for the complete text)
messages: list[dict[str, str]] = []

add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=500,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        print(text, end="")

    # Get the complete message for database storage
    # Get usage, stop_reason... etc
    final_message = stream.get_final_message()
    print("\n\n", final_message)

In [ ]:
import json

messages: list[dict[str, str]] = []

add_user_message(messages, "Generate a very short event bridge rule as json")

text = chat(messages)
print(text.text)

print("\n\n")

add_assistant_message(messages, "```json")

text = chat(messages, stop_sequences=["```"])
json_clean = json.loads(text.text.strip())  # validates; raises JSONDecodeError if invalid
print(json.dumps(json_clean, indent=2))                 # for display / copy-paste
print(json.dumps(json_clean, separators=(",", ":")))    # for CLI or API use


In [ ]:
# PENDING TO REVIEW ********
import os
from pathlib import Path
from dotenv import find_dotenv, dotenv_values

print("kernel cwd:", Path.cwd())
found = find_dotenv(usecwd=True)
print(".env found:", found or "NONE")
print("names in it:", list(dotenv_values(found)) if found else [])

k = os.environ.get("ANTHROPIC_API_KEY")
print("ANTHROPIC_API_KEY set:", bool(k))
if k:
    print("  starts with:", k[:7], "| length:", len(k), "| stray whitespace:", k != k.strip())